**Eye-Tracking & Neurocognitive Intelligence | Eye-Movement Modelling & Advanced AI | Presented by: Reza Saadatyar (2026)**

**05 | Prospective Main-Cohort Final Model Selection and Test**

![Eye-Movement Modelling independent participant Test evaluation and evidence integration](../../images/05_robustness_and_final_evaluation.svg)

**Next main execution (not yet run on 100 people):** This notebook reads the current GazeBase Round 1 / Session 1 study with participant count configured once in Notebook 01 (100 by default, optionally 300 for a separate, frozen protocol). It classifies the FXS/HSS/RAN/TEX original task identity, **not** cognition or a clinical outcome.

The introductory figure and previous committed scores describe the **earlier ten-person pilot** and are not main-study results. Use the matching stage ZIPs in /content in any fresh Colab Runtime: explicitly uploaded archives take precedence over stale pilot files checked out from GitHub. Independent-person roles and the main-cohort count are checked. No Google Drive mount is involved. Test is evaluated only after freezing the primary choice using the current Validation results.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_movement_modelling")

# ------------------------- 2. Scientific Computation -------------------------
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from IPython.display import display

# --------------------- 3. Research and Output Utilities ----------------------
from s2_handoff import ensure_s2_stage_inputs, verify_main_cohort
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab
from s2_data import (
    TASKS, load_protocol, load_windows, verify_temporal_source_alignment,
)
from s2_metrics import (
    FEATURE_COLUMN_NAMES, CLASS_TO_INDEX, metric_row,
    classical_record_predictions
)
from s2_training import (
    load_temporal, predict_windows, device_for_training,
)

# ----------------------------- 4. Output Folder ------------------------------
output_dir = root / "eye_movement_modelling" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Eye-Movement Modelling final-stage outputs:", output_dir)


**02. Recover Frozen Compatible Upstream Outputs**

Restore the matching stage-01, 02, 03 and 04 ZIPs when using a fresh Runtime. Check main-cohort counts, source-record identity, separate participant roles and agreement of all saved Validation metrics. Do not use stale pilot checkpoints.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# ==================== 02. Recover Frozen Producer Outputs ====================

# -------------------- 1. Restore Explicit Current Inputs ---------------------
ensure_s2_stage_inputs(output_dir, 1, (
    "01_record_features.csv", "01_split_assignments.csv",
    "01_sequence_windows.npz", "01_protocol.csv",
))
ensure_s2_stage_inputs(output_dir, 2, (
    "02_validation_metrics.csv", "02_class_prior.joblib",
    "02_logistic_regression.joblib", "02_random_forest.joblib",
    "02_gradient_boosting.joblib",
    "02_class_prior_validation_predictions.csv",
    "02_logistic_regression_validation_predictions.csv",
    "02_random_forest_validation_predictions.csv",
    "02_gradient_boosting_validation_predictions.csv",
))
ensure_s2_stage_inputs(output_dir, 3, (
    "03_neural_validation_metrics.csv", "03_cnn.pt", "03_tcn.pt",
    "03_cnn_validation_predictions.csv",
    "03_tcn_validation_predictions.csv",
))
ensure_s2_stage_inputs(output_dir, 4, (
    "04_neural_validation_metrics.csv", "04_gru.pt",
    "04_transformer.pt",
    "04_gru_validation_predictions.csv",
    "04_transformer_validation_predictions.csv",
))

# ------------------- 2. Load Matching Protocol and Metrics -------------------
protocol = load_protocol(output_dir / "01_protocol.csv")
split = pd.read_csv(
    output_dir / "01_split_assignments.csv",
    dtype={"participant_id": str, "record_id": str},
)
features = pd.read_csv(
    output_dir / "01_record_features.csv",
    dtype={"participant_id": str, "record_id": str},
)
arrays = load_windows(output_dir / "01_sequence_windows.npz")
role_counts = verify_main_cohort(protocol, split)
print("Main cohort participant roles:", role_counts)
candidate_order = (
    "class_prior", "logistic_regression", "random_forest",
    "gradient_boosting", "cnn", "tcn", "gru", "transformer",
)
validation = pd.concat([
    pd.read_csv(output_dir / "02_validation_metrics.csv"),
    pd.read_csv(output_dir / "03_neural_validation_metrics.csv"),
    pd.read_csv(output_dir / "04_neural_validation_metrics.csv"),
], ignore_index=True)

# --------------------------- 3. Verify Comparison ----------------------------
if len(validation) != len(candidate_order) or (
    set(validation["model"]) != set(candidate_order)
) or validation["model"].duplicated().any():
    raise ValueError("Eight distinct saved Validation models are required")
if not validation["partition"].eq("validation").all():
    raise ValueError("Validation selection cannot use Test scores")
if validation["n_records"].nunique() != 1 or (
    validation["n_participants"].nunique() != 1
):
    raise ValueError("Validation models used different evaluation units")
actual_val = split.loc[split.split.eq("validation")]
if not validation.n_records.eq(len(actual_val)).all() or not (
    validation.n_participants.eq(actual_val.participant_id.nunique()).all()
):
    raise ValueError(
        "Saved model Validation scores are incompatible with the current "
        "source participant split. Rerun dependent model notebooks."
    )
# ----------------- 4. Cross-Check Original Validation Identities -----------
# Identical counts do not guarantee the same Validation people after a
# rerun. Confirm the exact source record IDs and targets for every model.
expected_val_ids = set(actual_val.record_id.astype(str))
target_lookup = dict(zip(
    actual_val.record_id.astype(str),
    actual_val.task.map(CLASS_TO_INDEX),
))
for name, stage in (
    ("class_prior", "02"), ("logistic_regression", "02"),
    ("random_forest", "02"), ("gradient_boosting", "02"),
    ("cnn", "03"), ("tcn", "03"),
    ("gru", "04"), ("transformer", "04"),
):
    pred = pd.read_csv(
        output_dir / f"{stage}_{name}_validation_predictions.csv",
        dtype={"record_id": str},
    )
    if (
        pred.record_id.duplicated().any()
        or set(pred.record_id) != expected_val_ids
        or not pred.partition.eq("validation").all()
    ):
        raise ValueError(
            f"Validation source records mismatch for {name}; "
            "do not mix model checkpoints from different study runs."
        )
    expected_targets = pred.record_id.map(target_lookup).astype(int)
    if not np.array_equal(
        pred.target_index.to_numpy(dtype=int), expected_targets.to_numpy()
    ):
        raise ValueError(
            f"Validation class labels mismatch for {name}; "
            "rebuild the model outputs from the current cohort."
        )

alignment_counts = verify_temporal_source_alignment(arrays, split)
print("Aligned sequence windows by group:", alignment_counts)
print("Study mode:", protocol["purpose"])
print("Validation independent participants:",
      validation.n_participants.iloc[0])
display(validation[[
    "model", "n_records", "macro_f1", "balanced_accuracy", "accuracy",
]])


**03. Choose Primary Model from Validation**

Use only current-cohort Validation Macro F1, Balanced Accuracy and the fixed candidate priority to select the primary model before opening Test. Do not assume the historical pilot's Logistic Regression choice will recur.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# =================== 03. Freeze Validation-Based Selection ===================

# --------------------- 1. Apply A Priori Decision Order ----------------------
priority = {name: i for i, name in enumerate(candidate_order)}
selection = validation.copy()
selection["tie_priority"] = selection.model.map(priority)
selection = selection.sort_values(
    ["macro_f1", "balanced_accuracy", "tie_priority"],
    ascending=[False, False, True],
).reset_index(drop=True)
selected_name = str(selection.loc[0, "model"])
decision = {
    "selected_model": selected_name,
    "selection_rule": "validation_macro_f1_then_balanced_accuracy_then_order",
    "purpose": protocol["purpose"],
    "test_used_in_selection": False,
    "validation_n_people": int(selection.n_participants.iloc[0]),
}
pd.DataFrame([decision]).to_csv(
    output_dir / "05_model_selection.csv", index=False,
)
(output_dir / "05_model_selection.json").unlink(missing_ok=True)
print("Validation-selected model:", selected_name)
display(selection[["model", "macro_f1", "balanced_accuracy"]])

# -------------------------- 2. Visualise Selection ---------------------------
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(selection["model"], selection["macro_f1"])
ax.set(ylabel="Record-level Validation Macro F1", ylim=(0, 1),
       title="Model Selection on Validation Only")
ax.tick_params(axis="x", rotation=28)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "05_validation_selection.png")
plt.show()


**04. Score Reserved Main Test Once**

After finalising the current Validation selection, evaluate frozen classical and temporal models on each original Test record. Aggregate temporal windows within records. These Test scores must not be used to change models or participant selection.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# ====================== 04. Held-Out Record Evaluation =======================

# ------------------- 1. Verify Test Source Record Identity -------------------
test = features.loc[features.split.eq("test")].copy()
expected = set(split.loc[split.split.eq("test"), "record_id"])
if test.empty or set(test.record_id) != expected:
    raise ValueError("Test record identities missing or incompatible")
test_participants = test.participant_id.nunique()
print("Held-out people:", test_participants)
print("Held-out original task records:", len(test))

# ----------------------- 2. Score Classical Pipelines ------------------------
predictions = []
for name in candidate_order[:4]:
    model = joblib.load(output_dir / f"02_{name}.joblib")
    pred = classical_record_predictions(
        model, test, model_name=name, partition="test",
    )
    predictions.append(pred)

# ---------------------- 3. Score Frozen Temporal Models ----------------------
membership = dict(zip(split.record_id.astype(str), split.split))
record_roles = np.asarray([
    membership[str(r)] for r in arrays["record_id"]
])
test_mask = record_roles == "test"
for name, stage in (("cnn", "03"), ("tcn", "03"),
                    ("gru", "04"), ("transformer", "04")):
    model, state = load_temporal(output_dir / f"{stage}_{name}.pt")
    proba = predict_windows(
        model, arrays["x"][test_mask], device=device_for_training(),
    )
    from s2_metrics import record_probabilities
    pred = record_probabilities(
        arrays["record_id"][test_mask],
        arrays["y"][test_mask], proba,
    )
    pred["model"] = name
    pred["partition"] = "test"
    pred = pred.merge(
        test[["record_id", "participant_id", "task"]],
        on="record_id", how="inner", validate="one_to_one",
    )
    predictions.append(pred)

# ----------------- 4. Persist Comparable Source-Level Scores -----------------
all_predictions = pd.concat(predictions, ignore_index=True)
if any(set(p.record_id) != expected for p in predictions):
    raise ValueError("Not all frozen models scored the same Test records")
all_predictions.to_csv(output_dir / "05_test_predictions.csv", index=False)
results = []
for name, frame in all_predictions.groupby("model", sort=False):
    result = metric_row(
        frame.target_index, frame.pred_index, model=name,
        partition="test", n_participants=test_participants,
    )
    result["selected_primary"] = bool(name == selected_name)
    result["study_purpose"] = protocol["purpose"]
    results.append(result)
test_metrics = pd.DataFrame(results).sort_values("model")
test_metrics.to_csv(output_dir / "05_test_metrics.csv", index=False)
display(test_metrics)


**05. Inspect Independent Test People**

Report the new primary confusion matrix and per-participant error patterns. Inspect independent Test-person denominators; do not copy the pilot's 7/8 result into this study.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# ================= 05. Inspect Frozen Primary Test Evidence ==================

# ----------------- 1. Display Primary Model Confusion Matrix -----------------
primary = all_predictions.loc[
    all_predictions.model.eq(selected_name)
].copy()
matrix = confusion_matrix(
    primary.target_index, primary.pred_index,
    labels=range(len(TASKS)),
)
fig, ax = plt.subplots(figsize=(6.4, 5.1))
ConfusionMatrixDisplay(
    matrix, display_labels=TASKS,
).plot(ax=ax, colorbar=False, values_format="d")
ax.set_title(f"Frozen Primary Test: {selected_name}")
fig.tight_layout()
save_figure(fig, output_dir / "05_selected_test_confusion.png")
plt.show()

# ----------------- 2. Describe Independent Test Participants -----------------
individual = []
for (name, person), group in all_predictions.groupby(
    ["model", "participant_id"], sort=True,
):
    row = metric_row(
        group.target_index, group.pred_index,
        model=name, partition="test_person",
        n_participants=1,
    )
    row["participant_id"] = person
    individual.append(row)
groupwise = pd.DataFrame(individual)
groupwise.to_csv(output_dir / "05_groupwise_test_metrics.csv", index=False)
display(groupwise[[
    "model", "participant_id", "n_records", "macro_f1",
    "balanced_accuracy", "accuracy",
]].head(20))


**06. Validation-Only Sensitivity**

Apply the existing defined pupil-feature or relative x/y masking to Validation only, without refitting. This remains a descriptive intervention, not proof of feature irrelevance or explanation fidelity.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# ================= 06. Inspect Validation-Only Perturbations =================

# --------------------- 1. Select Validation Source Units ---------------------
valid_records = features.loc[features.split.eq("validation")].copy()
base = validation.loc[validation.model.eq(selected_name)].iloc[0]
robust = [{
    "model": selected_name, "condition": "original_validation",
    "macro_f1": float(base.macro_f1),
    "n_records": int(base.n_records),
}]

# ------------------ 2. Apply One Defined Input Intervention ------------------
if selected_name in candidate_order[:4]:
    model = joblib.load(output_dir / f"02_{selected_name}.joblib")
    altered = valid_records.copy()
    altered[["pupil_coverage", "pupil_median_uncalibrated"]] = np.nan
    pred = classical_record_predictions(
        model, altered, model_name=selected_name,
        partition="validation",
    )
    score = metric_row(
        pred.target_index, pred.pred_index, model=selected_name,
        partition="validation", n_participants=valid_records.participant_id.nunique(),
    )
    condition = "pupil_features_masked"
elif selected_name in candidate_order[4:]:
    stage = "03" if selected_name in ("cnn", "tcn") else "04"
    model, _ = load_temporal(output_dir / f"{stage}_{selected_name}.pt")
    valid_mask = record_roles == "validation"
    altered = arrays["x"][valid_mask].copy()
    altered[:, :, 0:2] = 0
    proba = predict_windows(
        model, altered, device=device_for_training(),
    )
    from s2_metrics import record_probabilities
    pred = record_probabilities(
        arrays["record_id"][valid_mask],
        arrays["y"][valid_mask], proba,
    )
    score = metric_row(
        pred.target_index, pred.pred_index, model=selected_name,
        partition="validation", n_participants=valid_records.participant_id.nunique(),
    )
    condition = "relative_xy_channels_masked"
else:
    raise ValueError("Unknown selected model")
robust.append({
    "model": selected_name, "condition": condition,
    "macro_f1": score["macro_f1"], "n_records": score["n_records"],
})
robustness = pd.DataFrame(robust)
robustness.to_csv(
    output_dir / "05_robustness_validation.csv", index=False,
)
display(robustness)

# ------------------------- 3. Save Robustness Figure -------------------------
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(robustness.condition, robustness.macro_f1)
ax.set(ylabel="Validation Macro F1", ylim=(0, 1),
       title="Selected Model: Descriptive Input Perturbation")
ax.tick_params(axis="x", rotation=12)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "05_validation_robustness.png")
plt.show()


**07. Describe the Frozen Candidate Comparison**

Summarise new Test scores descriptively while keeping the Validation-selected primary model fixed. Task-stimulus confounding and dataset boundaries limit the claims.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# ==================== 07. Display Frozen Test Comparison =====================

# ------------------------- 1. Common Record Metrics --------------------------
ordered = test_metrics.set_index("model").reindex(candidate_order)
fig, ax = plt.subplots(figsize=(10.5, 4.8))
pos = np.arange(len(ordered))
ax.bar(pos - 0.17, ordered.macro_f1, width=0.34,
       label="Macro F1")
ax.bar(pos + 0.17, ordered.balanced_accuracy, width=0.34,
       label="Balanced accuracy")
ax.set_xticks(pos, ordered.index, rotation=25, ha="right")
ax.set(ylabel="Test score by source record", ylim=(0, 1),
       title="Frozen Models: Exploratory Test Comparison")
ax.grid(axis="y", alpha=0.2)
ax.legend()
fig.tight_layout()
save_figure(fig, output_dir / "05_final_model_comparison.png")
plt.show()
print("Preselected model:", selected_name)
print("Study purpose:", protocol["purpose"])
print("Test participants:", test_participants)


**08. Preserve Outputs and Export ZIP**

Save the current main-cohort selection, Test predictions and person-level metrics, Validation-only perturbation and figures in one stage-05 ZIP. This newly examined Test cannot be reused as untouched evidence in a later adaptive 300-person run.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# ====================== 08. Export Final-Stage Evidence ======================

# ------------------------ 1. Package Current Results -------------------------
# The helper prints one file list and the ZIP location.
download_outputs_from_colab("eye_movement_modelling/output", "05")
